In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_parquet('../data/steam_reviews_sampled.parquet')

print(df.shape)
df.head()

(2609677, 23)


,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,...,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,483066,70,Half-Life,67466662,english,games lit,1587088761,1587088761,True,0,...,True,False,False,76561198393674381,70,7,468.00,0.00,327.00,"1,587,146,640.00"
1,482926,70,Half-Life,67630987,english,fun game,1587307934,1587307934,True,0,...,True,False,False,76561198124244032,40,1,"4,384.00",0.00,"4,375.00","1,587,337,064.00"
2,491793,70,Half-Life,60621817,english,Classic,1577393132,1577393132,True,0,...,True,False,False,76561198184787194,55,2,"1,942.00",0.00,"1,122.00","1,577,485,023.00"
3,473102,70,Half-Life,81085377,turkish,Desinc izlemeden oynamayın.,1606667257,1606667257,True,0,...,True,False,False,76561198211172345,242,64,"1,485.00",0.00,"1,411.00","1,608,133,624.00"
4,476284,70,Half-Life,76039982,english,""" Why do we have tower these ridicoulous ties ? """,1600206820,1600206820,True,0,...,True,False,False,76561198352005275,23,3,"3,340.00",0.00,"2,883.00","1,602,516,534.00"


In [8]:
df.dtypes

Unnamed: 0                          int64
app_id                              int64
app_name                              str
review_id                           int64
language                              str
review                                str
timestamp_created                   int64
timestamp_updated                   int64
recommended                          bool
votes_helpful                       int64
votes_funny                         int64
weighted_vote_score               float64
comment_count                       int64
steam_purchase                       bool
received_for_free                    bool
written_during_early_access          bool
author.steamid                      int64
author.num_games_owned              int64
author.num_reviews                  int64
author.playtime_forever           float64
author.playtime_last_two_weeks    float64
author.playtime_at_review         float64
author.last_played                float64
dtype: object

In [9]:
df.isnull().sum() 

Unnamed: 0                           0
app_id                               0
app_name                             0
review_id                            0
language                             0
review                            4084
timestamp_created                    0
timestamp_updated                    0
recommended                          0
votes_helpful                        0
votes_funny                          0
weighted_vote_score                  0
comment_count                        0
steam_purchase                       0
received_for_free                    0
written_during_early_access          0
author.steamid                       0
author.num_games_owned               0
author.num_reviews                   0
author.playtime_forever              0
author.playtime_last_two_weeks       0
author.playtime_at_review         3058
author.last_played                   0
dtype: int64

**Recommendation** - ~87% positive, ~13% negative. Highly skewed toward positive.

**Playtime** - Measured in minutes. Median is 1,880 but mean is 8,827, indicating extreme right skew. A few users with millions of minutes of playtime are pulling the mean up.

In [10]:
print(df['recommended'].value_counts(normalize=True))
print(df['author.playtime_at_review'].describe())

recommended
True    0.87
False   0.13
Name: proportion, dtype: float64
count   2,606,619.00
mean        8,827.39
std        23,991.73
min             1.00
25%           559.00
50%         1,880.00
75%         6,841.00
max     2,071,808.00
Name: author.playtime_at_review, dtype: float64


# Purchasing Context

* 77% purchased through Steam, 23% obtained elsewhere (external key sites like Humble Bundle, etc.)
* Only 3% received for free. A small group, but exactly the one to test for bias later. Do free copy recipients rate more positively?
* 9% reviewed during early access, also a bias test candidate. Are early access reviewers more lenient or more harsh?

In [11]:
print(df['steam_purchase'].value_counts(normalize=True),"\n")
print(df['received_for_free'].value_counts(normalize=True),"\n")
print(df['written_during_early_access'].value_counts(normalize=True),"\n")


steam_purchase
True    0.77
False   0.23
Name: proportion, dtype: float64 

received_for_free
False   0.97
True    0.03
Name: proportion, dtype: float64 

written_during_early_access
False   0.91
True    0.09
Name: proportion, dtype: float64 





**votes_helpful** - 75% of reviews have 0 or 1 helpful votes. Most reviews get no engagement at all. The mean is ~2 but the max is 16,960. Extremely sparse, confirming this column isn't very useful for this branch.

**author.num_games_owned** - The mean is 1.6 million and the max is 4.3 trillion. Nobody owns 4 trillion games. There are likely corrupted or garbage values in this column. These outliers need investigation, probably cap or drop rows with unrealistic values.

**author.num_reviews** - Median reviewer has written 4 reviews, 75th percentile is 10. Most people don't write many reviews, but some have written 5,000+. Heavy right skew again.


In [12]:
print(df['votes_helpful'].describe(),"\n")
print(df['author.num_games_owned'].describe(),"\n")
print(df['author.num_reviews'].describe(),"\n")

count   2,609,677.00
mean            1.89
std            40.77
min             0.00
25%             0.00
50%             0.00
75%             1.00
max        16,960.00
Name: votes_helpful, dtype: float64 

count           2,609,677.00
mean            1,685,416.53
std         2,722,490,913.14
min                     0.00
25%                    22.00
50%                    61.00
75%                   145.00
max     4,398,046,511,170.00
Name: author.num_games_owned, dtype: float64 

count   2,609,677.00
mean           10.51
std            34.13
min             1.00
25%             2.00
50%             4.00
75%            10.00
max         5,236.00
Name: author.num_reviews, dtype: float64 



The first value (~4.4 trillion) is clearly corrupted data. The remaining values in the 17,000-22,000 range look plausible for heavy collectors on Steam.

In [13]:
df['author.num_games_owned'].nlargest(10)

1025764    4398046511170
2481502            22019
415023             21979
2282678            19024
941059             19022
1097449            19022
1377836            19022
1335279            18664
1879647            17343
2454041            17343
Name: author.num_games_owned, dtype: int64

**Cleaning `author.num_games_owned`**

Only 1 row exceeded the 25,000 threshold and was removed. After filtering, the distribution looks reasonable:

The median user owns 61 games, and the 75th percentile sits at 145. The max of ~22,000 is high but plausible for heavy collectors.

In [14]:
print((df['author.num_games_owned'] > 25000).sum())
df = df[df['author.num_games_owned'] <= 25000]
df['author.num_games_owned'].describe()

1


count   2,609,676.00
mean          132.67
std           280.06
min             0.00
25%            22.00
50%            61.00
75%           145.00
max        22,019.00
Name: author.num_games_owned, dtype: float64